# Solutions · 3. Monotone fits and the pool-adjacent-violators algorithm

Worked solutions to the three exercises of [Lecture 3](../../notebooks/part2_shape/03_thermodynamic_stability_and_pava_isotonic_regression.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: 4-Point Pen-and-Paper PAVA Walkthrough and a noisy synthetic isotherm")
y_micro = np.array([1.0, 3.0, 2.0, 4.5])
f_micro = linear_physics.pava_monotone_regression(y_micro, increasing=True)
print("4-Point Micro-Example Input y :", y_micro)
print("4-Point PAVA Increasing Output:", f_micro, "(Adjacent violators [3.0, 2.0] pooled to mean 2.5)")
assert np.allclose(f_micro, [1.0, 2.5, 2.5, 4.5], atol=1e-15)

# A synthetic isotherm of cubic-equation-of-state form with assumed constants (V_m in L/mol, P in MPa):
rng = np.random.default_rng(42)
Vm = np.linspace(0.12, 0.80, 40)
P_true = 33.2 / (Vm - 0.065) - 9.4 / (Vm ** 2 + 0.13 * Vm)
P_noisy = P_true + rng.normal(0.0, 2.2, size=len(Vm))
P_pava = linear_physics.pava_monotone_regression(P_noisy, increasing=False)

assert float(np.max(np.diff(P_pava))) <= 1e-15, "PAVA isotherm must satisfy dP/dVm <= 0 everywhere!"
assert np.isclose(np.mean(P_pava), np.mean(P_noisy), rtol=1e-14), "PAVA must conserve the sample mean!"
print(f"Noisy isotherm stability violations (dP > 0): {int(np.sum(np.diff(P_noisy) > 0))} intervals")
print(f"PAVA isotherm stability violations  (dP > 0): {int(np.sum(np.diff(P_pava) > 0))} intervals")
heading("Demonstration 2: Step-by-Step Stack Block-Pooling Inspection")
y_sub8 = P_noisy[:8]
blocks = []
for idx, val in enumerate(y_sub8):
    blocks.append([float(val), 1])
    while len(blocks) >= 2 and (blocks[-2][0] / blocks[-2][1]) < (blocks[-1][0] / blocks[-1][1]):  # decreasing check
        last = blocks.pop()
        blocks[-1][0] += last[0]
        blocks[-1][1] += last[1]
    plateaus = [round(b[0] / b[1], 2) for b in blocks]
    counts = [b[1] for b in blocks]
    print(f"Step {idx+1} (insert P={val:6.2f} MPa) -> Active blocks (mean, count): {list(zip(plateaus, counts))}")
heading("Demonstration 3: PAVA vs. Unconstrained Degree-7 Polynomial Benchmark")
poly_coef = np.polyfit(Vm, P_noisy, deg=7)
P_poly7 = np.polyval(poly_coef, Vm)
poly_violations = int(np.sum(np.diff(P_poly7) > 0))

rmse_noisy = float(np.sqrt(np.mean((P_noisy - P_true) ** 2)))
rmse_poly7 = float(np.sqrt(np.mean((P_poly7 - P_true) ** 2)))
rmse_pava = float(np.sqrt(np.mean((P_pava - P_true) ** 2)))
print(f"RMSE vs True Isotherm: Noisy = {rmse_noisy:.3f} MPa | Poly-7 = {rmse_poly7:.3f} MPa ({poly_violations} violations) | PAVA = {rmse_pava:.3f} MPa (0 violations)")
assert rmse_pava < rmse_noisy and poly_violations > 0
heading("Demonstration 4: The van der Waals loop and the equal-area plateau")
# Reduced van der Waals isotherm at T_r = 0.9:  P_r = 8 T_r / (3 V_r - 1) - 3 / V_r^2
T_r = 0.9
Vm_sub = np.linspace(0.45, 5.0, 1821)                      # reduced volume, uniform grid
P_vdw = 8.0 * T_r / (3.0 * Vm_sub - 1.0) - 3.0 / Vm_sub ** 2
P_maxwell_pava = linear_physics.pava_monotone_regression(P_vdw, increasing=False)
n_spinodal = int(np.sum(np.diff(P_vdw) > 0))
n_after = int(np.sum(np.diff(P_maxwell_pava) > 0))
plateau_vals, plateau_counts = np.unique(np.round(P_maxwell_pava, 9), return_counts=True)
p_sat_est = plateau_vals[int(np.argmax(plateau_counts))]

# Independent check: Maxwell's equal-area rule, solved by bisection on the saturation pressure.
def equal_area_mismatch(p_sat):
    """Integral of (P - p_sat) dV between the outer roots of P(V) = p_sat; zero at saturation."""
    roots = np.roots([3.0 * p_sat, -(p_sat + 8.0 * T_r), 9.0, -3.0])       # cubic in V_r
    v_l, v_g = np.min(roots.real), np.max(roots.real)
    antiderivative = lambda v: (8.0 * T_r / 3.0) * np.log(3.0 * v - 1.0) + 3.0 / v
    return antiderivative(v_g) - antiderivative(v_l) - p_sat * (v_g - v_l), v_l, v_g

p_lo, p_hi = 0.45, 0.72                                     # between the two spinodal pressures
for _ in range(60):
    p_mid = 0.5 * (p_lo + p_hi)
    if equal_area_mismatch(p_mid)[0] > 0.0:
        p_lo = p_mid
    else:
        p_hi = p_mid
p_sat_maxwell, v_liq, v_gas = p_mid, *equal_area_mismatch(p_mid)[1:]
on_plateau = np.isclose(P_maxwell_pava, p_sat_est, atol=1e-8)
print(f"Intervals where the pressure rises with the volume: {n_spinodal} -> after PAVA: {n_after}")
print(f"Plateau of the monotone fit    : P_r = {p_sat_est:.5f}, from V_r = {Vm_sub[on_plateau][0]:.3f} to {Vm_sub[on_plateau][-1]:.3f} ({np.max(plateau_counts)} grid points)")
print(f"Maxwell's equal-area rule      : P_r = {p_sat_maxwell:.5f}, from V_r = {v_liq:.3f} to {v_gas:.3f}")
assert n_spinodal > 10 and n_after == 0
assert np.isclose(np.mean(P_maxwell_pava), np.mean(P_vdw), rtol=1e-12)
assert abs(p_sat_est - p_sat_maxwell) < 2e-3
heading("Inside the Algorithm: By-Hand Stack PAVA vs. the Library")
def pava_increasing_by_hand(y_arr: np.ndarray) -> np.ndarray:
    st: list[list[float]] = []
    for v in y_arr:
        st.append([float(v), 1.0])
        while len(st) >= 2 and st[-2][0] / st[-2][1] > st[-1][0] / st[-1][1]:
            s_r, c_r = st.pop()
            st[-1][0] += s_r
            st[-1][1] += c_r
    return np.concatenate([np.full(int(c), s / c) for s, c in st])

p_hand = -pava_increasing_by_hand(-P_noisy)
diff_pava = float(np.max(np.abs(p_hand - P_pava)))
print("by hand (pure NumPy) first 6 pts:", np.round(p_hand[:6], 5))
print("library (engml_2_applied)      first 6 pts:", np.round(P_pava[:6], 5))
print(f"Max digit-for-digit discrepancy : {diff_pava:.2e}")
assert diff_pava < 1e-15
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Generate $N = 50$ noisy Langmuir $\mathrm{CO_2}$ adsorption uptake measurements $q(P) = q_{\max} K P / (1 + K P)$ of Langmuir form with assumed constants ($q_{\max} = 5.2\;\mathrm{mol/kg}, K = 1.4\;\mathrm{bar}^{-1}$) across $P \in [0.1, 10.0]\;\mathrm{bar}$ with Gaussian noise $\sigma = 0.25\;\mathrm{mol/kg}$. Apply `pava_monotone_regression(.., increasing=True)` and verify that (1) $\min(\Delta \hat{q}) \ge 0$, (2) $\mathrm{mean}(\hat{q}) = \mathrm{mean}(y)$, and (3) $\|\hat{\mathbf{q}}_{\mathrm{PAVA}} - \mathbf{q}_{\mathrm{true}}\|_2 < \|\mathbf{y} - \mathbf{q}_{\mathrm{true}}\|_2$.

In [2]:
heading("Exercise 1: PAVA Projection of a Synthetic CO2 Adsorption Isotherm")
rng_ex = np.random.default_rng(77)
P_ads = np.linspace(0.1, 10.0, 50)
q_true = 5.2 * 1.4 * P_ads / (1.0 + 1.4 * P_ads)
q_noisy = q_true + rng_ex.normal(0.0, 0.25, size=50)
q_pava = linear_physics.pava_monotone_regression(q_noisy, increasing=True)

assert float(np.min(np.diff(q_pava))) >= -1e-15
assert np.isclose(np.mean(q_pava), np.mean(q_noisy), rtol=1e-14)
assert np.linalg.norm(q_pava - q_true) < np.linalg.norm(q_noisy - q_true)
print(f"Synthetic adsorption isotherm, L2 error: Raw = {np.linalg.norm(q_noisy - q_true):.3f} -> PAVA = {np.linalg.norm(q_pava - q_true):.3f} mol/kg")


Exercise 1: PAVA Projection of a Synthetic CO2 Adsorption Isotherm
Synthetic adsorption isotherm, L2 error: Raw = 1.866 -> PAVA = 0.820 mol/kg


**Discussion.** Thermodynamic adsorption stability requires $(\partial q / \partial P)_T \ge 0$. PAVA projects the noisy uptake curve onto the monotone increasing cone and reduces the $L_2$ error from $1.87$ to $0.82\;\mathrm{mol/kg}$.

---

## Exercise 2

**Problem.** Show why Newton's method for the inverse problem $P(V_m) = P_{\mathrm{target}}$ fails on a staircase: for the pooled van der Waals isotherm of Demonstration 4 (`P_maxwell_pava` on the grid `Vm_sub`), compute the finite-difference slope $\Delta \hat{P}/\Delta V_m$ on every interval and report the fraction of intervals where it is zero.

In [3]:
heading("Exercise 2: Staircase Zero-Derivative Audit on PAVA Plateaus")
dP_pava = np.diff(P_maxwell_pava) / np.diff(Vm_sub)
zero_slope_frac = float(np.mean(np.abs(dP_pava) < 1e-14))
print(f"Fraction of V_m intervals with zero slope (dP/dVm == 0) on PAVA staircase: {zero_slope_frac*100:.1f}%")
assert zero_slope_frac > 0.30


Exercise 2: Staircase Zero-Derivative Audit on PAVA Plateaus
Fraction of V_m intervals with zero slope (dP/dVm == 0) on PAVA staircase: 38.3%


**Discussion.** The grid `Vm_sub` covers the liquid, the two-phase region and the vapour. On the part pooled into the plateau the slope is exactly zero, and the printed fraction is the share of the grid that this part takes. Newton's step, $\Delta V_m = -(P - P_{\mathrm{target}})/(dP/dV_m)$, divides by the slope and is undefined there. A monotone fit that is to be inverted by a gradient method needs a non-zero, continuous derivative, which is the subject of Lecture 4.

---

## Exercise 3

**Problem.** Implement a **weighted PAVA** function `weighted_pava_increasing(y: np.ndarray, weights: np.ndarray) -> np.ndarray` that minimizes $\sum_{i=1}^N w_i (f_i - y_i)^2$ subject to $f_1 \le \dots \le f_N$ by accumulating `[weighted_sum, total_weight]` in each stack block. Verify that it conserves the weighted mean $\sum_i w_i \hat{f}_i = \sum_i w_i y_i$ and matches `pava_monotone_regression` when $w_i \equiv 1$.

In [4]:
heading("Exercise 3: Implement Inverse-Variance Weighted PAVA From Scratch")
def weighted_pava_increasing(y: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Weighted isotonic regression minimizing sum_i w_i (f_i - y_i)^2 subject to f_1 <= ... <= f_N."""
    yv = np.asarray(y, dtype=float).ravel()
    wv = np.asarray(weights, dtype=float).ravel()
    blocks: list[list] = []
    for val, wt in zip(yv, wv):
        blocks.append([float(wt * val), float(wt), 1])
        while len(blocks) >= 2 and (blocks[-2][0] / blocks[-2][1]) > (blocks[-1][0] / blocks[-1][1]):
            sw_r, w_r, c_r = blocks.pop()
            blocks[-1][0] += sw_r
            blocks[-1][1] += w_r
            blocks[-1][2] += c_r
    return np.concatenate([np.full(c, sw / w) for sw, w, c in blocks])

w_weights = np.linspace(1.0, 5.0, 50)
q_wpava = weighted_pava_increasing(q_noisy, w_weights)
q_unif = weighted_pava_increasing(q_noisy, np.ones(50))
assert np.all(np.diff(q_wpava) >= -1e-15)
assert np.isclose(np.sum(w_weights * q_wpava), np.sum(w_weights * q_noisy), rtol=1e-14)
assert np.allclose(q_unif, q_pava, atol=1e-14)
print("Weighted PAVA weighted-mean residual:", abs(float(np.sum(w_weights * (q_wpava - q_noisy)))))


Exercise 3: Implement Inverse-Variance Weighted PAVA From Scratch
Weighted PAVA weighted-mean residual: 2.90878432451791e-14


**Discussion.** Replacing uniform counts with inverse-variance weights $w_i = \sigma_i^{-2}$ generalizes PAVA to heteroscedastic thermodynamic datasets while preserving $O(N)$ complexity and exact weighted-mean conservation.